# End-to-End ML Pipeline for Customer Churn Prediction

## Objective

Develop a production-ready machine learning pipeline using Scikit-learn's Pipeline API to predict customer churn.

---

### Dataset
Telco Customer Churn Dataset

---

### Models Used

- Logistic Regression
- Random Forest

---

### Techniques Used

- Data Cleaning
- Feature Engineering
- Pipeline API
- ColumnTransformer
- GridSearchCV
- Model Evaluation
- Joblib Model Export

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Train-test split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Save model
import joblib

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

## 1. Load the Dataset

In this section, we load the Telco Customer Churn dataset and inspect its structure to understand the available features and target variable.

In [ ]:
# Load the dataset
df = pd.read_csv("Telco Churn.csv")

# Display the first 5 rows
df.head()

## 2. Dataset Overview

Let's examine the size, structure, and data types of the dataset.

In [ ]:
# Shape of the dataset
print("Dataset Shape:", df.shape)

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Dataset information
print("\nDataset Information:")
df.info()

## 3. Statistical Summary

Generate descriptive statistics for numerical and categorical features.

In [ ]:
# Numerical statistics
df.describe()

In [ ]:
# Statistics for categorical columns
df.describe(include="object")

## 4. Missing Values

Check whether the dataset contains missing values.

In [ ]:
# Missing values
missing_values = df.isnull().sum()

print(missing_values)

## 5. Duplicate Records

Check whether duplicate records are present in the dataset.

In [ ]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

## 6. Target Variable Distribution

Let's examine how many customers churned versus stayed.

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True) * 100

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x="Churn",
    data=df
)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Count")

plt.show()

In [ ]:
print(df.dtypes)

In [ ]:
for col in df.select_dtypes(include="object"):
    print(f"\n{col}")
    print(df[col].unique())

## 7. Separate Features and Target Variable

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [ ]:
# Remove extra spaces and convert to lowercase
df["Churn"] = df["Churn"].str.strip().str.lower()

# Map to numerical values
y = df["Churn"].map({
    "no": 0,
    "yes": 1
})

In [ ]:
# Numerical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns

# Categorical columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

## 8. Create a Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
print(df["Churn"].unique())

In [ ]:
print(y.isnull().sum())

In [ ]:
# Clean the Churn column
df["Churn"] = df["Churn"].str.strip().str.lower()

In [ ]:
# Recreate X and y
X = df.drop("Churn", axis=1)

y = df["Churn"].map({
    "no": 0,
    "yes": 1
})

In [ ]:
print(y.unique())

In [ ]:
print(y.isnull().sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training Set:", X_train.shape)
print("Testing Set:", X_test.shape)
print("Training Labels:", y_train.shape)
print("Testing Labels:", y_test.shape)

## 9. Logistic Regression Pipeline

In this section, we combine the preprocessing pipeline with a Logistic Regression classifier to create a complete machine learning pipeline.

In [ ]:
# Create Logistic Regression Pipeline
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000))
])

In [ ]:
# Train Logistic Regression
logistic_pipeline.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred_lr = logistic_pipeline.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_lr))

In [ ]:
print(classification_report(y_test, y_pred_lr))

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)

ConfusionMatrixDisplay(confusion_matrix=cm).plot()

plt.title("Logistic Regression Confusion Matrix")

plt.show()

## 10. Random Forest Pipeline

Build and evaluate a Random Forest classifier using the same preprocessing pipeline.

In [ ]:
# Create Random Forest Pipeline
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [ ]:
rf_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

In [ ]:
print(classification_report(y_test, y_pred_rf))

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

ConfusionMatrixDisplay(confusion_matrix=cm).plot()

plt.title("Random Forest Confusion Matrix")

plt.show()

## 11. Model Comparison

Compare the performance of Logistic Regression and Random Forest using accuracy.

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ]
})

comparison

## 12. Hyperparameter Tuning - Logistic Regression

Use GridSearchCV to find the optimal hyperparameters for the Logistic Regression model.

In [ ]:
# Logistic Regression Pipeline
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000))
])

In [ ]:
param_grid_lr = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__solver": ["liblinear", "lbfgs"]
}

In [ ]:
grid_lr = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid_lr,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(grid_lr.best_params_)

In [ ]:
print("Best Parameters:")
print(grid_lr.best_params_)

In [ ]:
best_lr = grid_lr.best_estimator_

y_pred_best_lr = best_lr.predict(X_test)

print(classification_report(y_test, y_pred_best_lr))

print("Accuracy:", accuracy_score(y_test, y_pred_best_lr))

In [ ]:
## 1. Hyperparameter Tuning - Random Forest

Optimize the Random Forest classifier using GridSearchCV.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [ ]:
param_grid_rf = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5]
}

In [ ]:
grid_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid_rf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(grid_rf.best_params_)

In [ ]:
print("Best Cross Validation Score:")
print(grid_rf.best_score_)

In [ ]:
best_rf = grid_rf.best_estimator_

y_pred_best_rf = best_rf.predict(X_test)

print(classification_report(y_test, y_pred_best_rf))

print("Accuracy:", accuracy_score(y_test, y_pred_best_rf))

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Tuned Logistic Regression",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_best_lr),
        accuracy_score(y_test, y_pred_best_rf)
    ]
})

comparison.sort_values(
    by="Accuracy",
    ascending=False
)

In [ ]:
joblib.dump(best_rf, "churn_pipeline.joblib")

In [ ]:
loaded_pipeline = joblib.load("churn_pipeline.joblib")

sample_prediction = loaded_pipeline.predict(X_test.iloc[:5])

print(sample_prediction)

In [ ]:
results = pd.DataFrame({
    "Actual": y_test.iloc[:5].values,
    "Predicted": sample_prediction
})

results["Actual"] = results["Actual"].map({0: "No Churn", 1: "Churn"})
results["Predicted"] = results["Predicted"].map({0: "No Churn", 1: "Churn"})

results

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Tuned Logistic Regression",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_best_lr),
        accuracy_score(y_test, y_pred_best_rf)
    ],
    "Precision": [
        classification_report(y_test, y_pred_lr, output_dict=True)["weighted avg"]["precision"],
        classification_report(y_test, y_pred_rf, output_dict=True)["weighted avg"]["precision"],
        classification_report(y_test, y_pred_best_lr, output_dict=True)["weighted avg"]["precision"],
        classification_report(y_test, y_pred_best_rf, output_dict=True)["weighted avg"]["precision"]
    ],
    "Recall": [
        classification_report(y_test, y_pred_lr, output_dict=True)["weighted avg"]["recall"],
        classification_report(y_test, y_pred_rf, output_dict=True)["weighted avg"]["recall"],
        classification_report(y_test, y_pred_best_lr, output_dict=True)["weighted avg"]["recall"],
        classification_report(y_test, y_pred_best_rf, output_dict=True)["weighted avg"]["recall"]
    ],
    "F1-Score": [
        classification_report(y_test, y_pred_lr, output_dict=True)["weighted avg"]["f1-score"],
        classification_report(y_test, y_pred_rf, output_dict=True)["weighted avg"]["f1-score"],
        classification_report(y_test, y_pred_best_lr, output_dict=True)["weighted avg"]["f1-score"],
        classification_report(y_test, y_pred_best_rf, output_dict=True)["weighted avg"]["f1-score"]
    ]
})

comparison.sort_values(by="Accuracy", ascending=False)